# CatDiT — Conditional Catalyst Generation (Demo)

Generate catalyst structures with two conditioning modes:

- **CatDiT-AB** — condition on **adsorbate + binding energy**
- **CatDiT-C** — condition on **catalyst class**

Set **Runtime → Change runtime type → GPU** before running.

## 1. Check GPU

In [1]:
!nvidia-smi

Sat May 30 07:50:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install dependencies and clone the repo

Uses Colab's default Python 3.12 + torch 2.11 + numpy 2 — no version downgrade.
Pymatgen / matminer latest are numpy-2 compatible; fairchem is excluded (not needed for inference).

In [2]:
import torch; print(torch.__version__)

2.11.0+cu128


In [3]:
!pip install -q torch-geometric torch_scatter \
    -f https://data.pyg.org/whl/torch-2.11.0+cu128.html

!pip install -q lightning==2.4.0 hydra-core==1.3.2 hydra-colorlog omegaconf==2.3.0 \
    e3nn pymatgen matminer pyxtal rootutils rich huggingface_hub ipywidgets \
    pandas seaborn scikit-learn svgwrite cairosvg reportlab importlib_resources

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 128.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 6.9 MB/s eta 0:00:00
   

In [4]:
%cd /content
!git clone https://github.com/doouv/CatDiT.git
%cd /content/CatDiT

/content
Cloning into 'CatDiT'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 173 (delta 22), reused 163 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (173/173), 8.23 MiB | 33.07 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/CatDiT


## 3. Download checkpoints and define the generator

Downloads CatDiT-AB, CatDiT-C, and the shared VAE-L from Hugging Face
(under `ldm/` and `vae/` folders in the model repo). `num_nodes_bincount.pt`
ships with the repo under `data/oc20/`.

In [16]:
import os, torch
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download
from src.models.ldm_module import LatentDiffusionLitModule
from src.eval.catalyst import Catalyst

REPO = "doouv/catalyst-diffusion-transformer"
ckpt_uncond = hf_hub_download(REPO, "ldm/CatDiT.ckpt")
ckpt_ab = hf_hub_download(REPO, "ldm/CatDiT-AB.ckpt")
ckpt_c = hf_hub_download(REPO, "ldm/CatDiT-C.ckpt")
vae_s = hf_hub_download(REPO, "vae/VAE-S.ckpt")
vae_l = hf_hub_download(REPO, "vae/VAE-L.ckpt")
bincount_path = "/content/CatDiT/data/oc20/num_nodes_bincount.pt"


def generate(ckpt, vae, out_dir, n=8, cfg_scale=2.0,
             ads_id=None, binding_energy=None, cat_class=None):
    """Load model, sample structures under the given conditions, save CIFs."""
    cg = OmegaConf.create({
        "binding_energy": {"use": binding_energy is not None, "value": binding_energy},
        "ads_id":         {"use": ads_id is not None,         "value": ads_id},
        "cat_class":      {"use": cat_class is not None,      "value": cat_class},
    })
    model = LatentDiffusionLitModule.load_from_checkpoint(
        ckpt, autoencoder_ckpt=vae, conditional_generation=cg,
        map_location="cuda", strict=False)
    model.eval(); model.setup(stage="test")
    bincount = torch.load(bincount_path, map_location="cuda")

    os.makedirs(out_dir, exist_ok=True)
    with torch.no_grad():
        out, batch, _ = model.sample_and_decode(
            num_nodes_bincount=bincount, batch_size=n, cfg_scale=cfg_scale)

    n_valid, idx0 = 0, 0
    for j, num_atom in enumerate(batch["num_atoms"].tolist()):
        atom_types = out["atom_types"].narrow(0, idx0, num_atom).argmax(dim=1)
        atom_types[atom_types == 0] = 1
        tags = out["tags"].narrow(0, idx0, num_atom).argmax(dim=1)
        frac = out["frac_coords"].narrow(0, idx0, num_atom)
        lengths = out["lengths"][j] * float(num_atom) ** (1 / 3)
        angles = torch.rad2deg(out["angles"][j])
        idx0 += num_atom
        cat = Catalyst({
            "atom_types": atom_types.cpu().numpy(), "tags": tags.cpu().numpy(),
            "frac_coords": frac.cpu().numpy(), "lengths": lengths.cpu().numpy(),
            "angles": angles.cpu().numpy(), "sample_idx": j})
        if cat.constructed and cat.struct_valid:
            cat.structure.to(filename=os.path.join(out_dir, f"structure_{j}.cif"))
            n_valid += 1
    print(f"Saved {n_valid}/{n} valid structures to {out_dir}/")

## 4. CatDiT — Unconditional generation

Set `num_samples` then run. After generation, a slider lets you step through the structures (rendered with ASE).

In [17]:
num_samples = 5 #@param {type:"integer"}
generate(ckpt_uncond, vae_s,"/content/out_uncond", n=num_samples)

Saved 5/5 valid structures to /content/out_uncond/


In [18]:
# step through generated structures with a slider (ASE rendering)
import glob
import matplotlib.pyplot as plt
from ase.io import read
from ase.visualize.plot import plot_atoms
from ipywidgets import interact, IntSlider

ab_atoms = [read(p) for p in sorted(glob.glob("/content/out_uncond/*.cif"))]

def show_uncond(i):
    fig, axes = plt.subplots(1, 4, figsize=(15, 5))
    views = [
        ("top",   "0x, 0y, 0z"),
        ("front", "-90x, 0y, 0z"),
        ("left",  "-90x, -90y, 0z"),
        ("perspective", "-75x, 45y, 10z")
    ]
    for ax, (name, rot) in zip(axes, views):
        ax.axis("off")
        ax.set_title(name)
        plot_atoms(ab_atoms[i], ax, radii=0.8, rotation=rot)
    fig.suptitle(f"Uncond · structure_{i}")
    plt.show()

interact(show_uncond, i=IntSlider(min=0, max=len(ab_atoms) - 1, value=0, description="idx"))

interactive(children=(IntSlider(value=0, description='idx', max=4), Output()), _dom_classes=('widget-interact'…

<function __main__.show_uncond(i)>

## 5. CatDiT-AB — adsorbate + binding energy

Set `ads_id`, `binding_energy` (eV), and `num_samples` below, then run.

**Adsorbates (`ads_id`)**

| `ads_id` | symbol | `ads_id` | symbol | `ads_id` | symbol | `ads_id` | symbol | `ads_id` | symbol |
|---:|---|---:|---|---:|---|---:|---|---:|---|
| 0 | `O` | 1 | `H` | 2 | `OH` | 3 | `OH2` | 4 | `C` |
| 5 | `CO` | 6 | `CH` | 7 | `CHO` | 8 | `COH` | 9 | `CH2` |
| 10 | `CH2O` | 11 | `CHOH` | 12 | `CH3` | 13 | `OCH3` | 14 | `CH2OH` |
| 15 | `CH4` | 16 | `OHCH3` | 17 | `CC` | 18 | `CCO` | 19 | `CCH` |
| 20 | `CHCO` | 21 | `CCHO` | 22 | `COCHO` | 23 | `CCHOH` | 24 | `CCH2` |
| 25 | `CHCH` | 26 | `CH2CO` | 27 | `CHCHO` | 29 | `COCH2O` | 30 | `CHOCHO` |
| 31 | `COHCHO` | 32 | `COHCOH` | 33 | `CCH3` | 34 | `CHCH2` | 35 | `COCH3` |
| 38 | `CHCHOH` | 39 | `CCH2OH` | 40 | `CHOCHOH` | 41 | `COCH2OH` | 42 | `COHCHOH` |
| 44 | `OCHCH3` | 45 | `COHCH3` | 46 | `CHOHCH2` | 47 | `CHCH2OH` | 48 | `OCH2CHOH` |
| 49 | `CHOCH2OH` | 50 | `COHCH2OH` | 51 | `CHOHCHOH` | 52 | `CH2CH3` | 53 | `OCH2CH3` |
| 54 | `CHOHCH3` | 55 | `CH2CH2OH` | 56 | `CHOHCH2OH` | 57 | `OHCH2CH3` | 58 | `NH2N(CH3)2` |
| 59 | `ONN(CH3)2` | 60 | `OHNNCH3` | 62 | `ONH` | 63 | `NHNH` | 65 | `NNH` |
| 67 | `NO2NO2` | 68 | `NNO` | 69 | `N2` | 70 | `ONNH2` | 71 | `NH2` |
| 72 | `NH3` | 73 | `NONH` | 74 | `NH` | 75 | `NO2` | 76 | `NO` |
| 77 | `N` | 78 | `NO3` | 79 | `OHNH2` | 80 | `ONOH` | 81 | `CN` |
| 82 | `HO2` | 83 | `O2` |  |  |  |  |  |  |

In [6]:
# CatDiT-AB conditions
ads_id = 0 #@param {type:"integer"}
binding_energy = -1.0 #@param {type:"number"}
num_samples = 5 #@param {type:"integer"}

generate(ckpt_ab, "/content/out_ab", n=num_samples,
         ads_id=ads_id, binding_energy=binding_energy)

/content/CatDiT/src/models/components/equiformer_v2/layer_norm.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/content/CatDiT/src/models/components/equiformer_v2/layer_norm.py:159: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/content/CatDiT/src/models/components/equiformer_v2/layer_norm.py:233: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/content/CatDiT/src/models/components/equiformer_v2/layer_norm.py:322: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


Saved 5/5 valid structures to /content/out_ab/


In [7]:

# step through generated structures with a slider (ASE rendering)
import glob
import matplotlib.pyplot as plt
from ase.io import read
from ase.visualize.plot import plot_atoms
from ipywidgets import interact, IntSlider

ab_atoms = [read(p) for p in sorted(glob.glob("/content/out_ab/*.cif"))]

def show_ab(i):
    fig, axes = plt.subplots(1, 4, figsize=(15, 5))
    views = [
        ("top",   "0x, 0y, 0z"),
        ("front", "-90x, 0y, 0z"),
        ("left",  "-90x, -90y, 0z"),
        ("perspective", "-75x, 45y, 10z")
    ]
    for ax, (name, rot) in zip(axes, views):
        ax.axis("off")
        ax.set_title(name)
        plot_atoms(ab_atoms[i], ax, radii=0.8, rotation=rot)
    fig.suptitle(f"AB · structure_{i}")
    plt.show()

interact(show_ab, i=IntSlider(min=0, max=len(ab_atoms) - 1, value=0, description="idx"))

interactive(children=(IntSlider(value=0, description='idx', max=4), Output()), _dom_classes=('widget-interact'…

<function __main__.show_ab(i)>

## 6. CatDiT-C — catalyst class

Set `cat_class` (0–4) and `num_samples` below, then run.


**Catalyst classes (`cc`)**
| `cat_class` | description |
|---:|---|
| 0 | intermetallic / unary |
| 1 | metalloid |
| 2 | non-metal |
| 3 | halide |
| 4 | oxide |

In [8]:
cat_class = 4 #@param {type:"integer"}
num_samples = 5 #@param {type:"integer"}

# CatDiT-C conditions
generate(ckpt_c, "/content/out_c", n=num_samples, cat_class=cat_class)

Saved 5/5 valid structures to /content/out_c/


In [9]:
# step through generated structures with a slider (ASE rendering)
import glob
import matplotlib.pyplot as plt
from ase.io import read
from ase.visualize.plot import plot_atoms
from ipywidgets import interact, IntSlider

c_atoms = [read(p) for p in sorted(glob.glob("/content/out_c/*.cif"))]

def show_c(i):
    fig, axes = plt.subplots(1, 4, figsize=(15, 5))
    views = [
        ("top",   "0x, 0y, 0z"),
        ("front", "-90x, 0y, 0z"),
        ("left",  "-90x, -90y, 0z"),
        ("perspective", "-75x, 45y, 10z")
    ]
    for ax, (name, rot) in zip(axes, views):
        ax.axis("off")
        ax.set_title(name)
        plot_atoms(c_atoms[i], ax, radii=0.8, rotation=rot)
    fig.suptitle(f"C · structure_{i}")
    plt.show()

interact(show_c, i=IntSlider(min=0, max=len(c_atoms) - 1, value=0, description="idx"))

interactive(children=(IntSlider(value=0, description='idx', max=4), Output()), _dom_classes=('widget-interact'…

<function __main__.show_c(i)>